# Near-Misses, Not Wrong Turns — real-model experiments with Gemma 4

Companion notebook for the Kaggle Paper Track writeup *Near-Misses, Not Wrong Turns: A Stratified Failure Taxonomy for SWE Coding Agents*.
Code, labels and rubric: https://github.com/San-Lins/swe-failure-taxonomy

This notebook replaces the stub demos from the paper with real runs:

1. **hygiene-check** on the 250 annotated agent patches (and their gold patches), validated against the human `HYG` labels.
2. **localization-probe** with **Gemma 4** (E4B and 12B, instruction-tuned) on the unique task instances behind the 250 failures: BM25 retrieves 50 candidate files from the *full* repository file list at `base_commit`, then Gemma re-ranks them. No editing is allowed.

Settings: Accelerator **GPU T4 ×2**, Internet **on**. Runs end to end with *Save & Run All*.

In [ ]:
%%capture
!pip install -q -U "transformers>=4.57" accelerate bitsandbytes rank_bm25 datasets kagglehub

In [ ]:
import os, re, json, time, subprocess, collections, math, random
import pandas as pd
WORK = "/kaggle/working"
os.makedirs(WORK, exist_ok=True)
random.seed(0)

# Released labels + prototype code from the paper repo
if not os.path.exists("/kaggle/working/swe-failure-taxonomy"):
    subprocess.run(["git", "clone", "--depth", "1",
                    "https://github.com/San-Lins/swe-failure-taxonomy", "/kaggle/working/swe-failure-taxonomy"], check=True)
REPO = "/kaggle/working/swe-failure-taxonomy"
labels = json.load(open(f"{REPO}/data/full_labels_v3.json"))
print(len(labels), "labeled trajectories")
import sys; sys.path.insert(0, f"{REPO}/eval-proto/hygiene-check"); sys.path.insert(0, f"{REPO}/eval-proto/localization-probe")
from checker import check            # hygiene checker, unchanged from the repo
from scorer import score_instance, aggregate

## 1. Load source data (SWE-Gym trajectories and task instances)

In [ ]:
from datasets import load_dataset, get_dataset_split_names
TRAJ = "SWE-Gym/OpenHands-Sampled-Trajectories"
split = get_dataset_split_names(TRAJ)
print("trajectory splits:", split)
traj = load_dataset(TRAJ, split=split[0])
print(traj)

gym = load_dataset("SWE-Gym/SWE-Gym", split="train")
gym = {r["instance_id"]: r for r in gym}
print(len(gym), "SWE-Gym instances")

In [ ]:
# Join labels to trajectories by row index, and check instance ids agree
rows, mismatch = [], 0
for L in labels:
    r = traj[int(L["idx"])]
    if r["instance_id"] != L["instance_id"]:
        mismatch += 1; continue
    tr = r["test_result"] or {}
    rows.append(dict(L, agent_patch=(tr.get("git_patch") or "")))
print(f"joined {len(rows)} / {len(labels)}; id mismatches: {mismatch}")
assert mismatch == 0, "row index -> instance_id mismatch; check split ordering"

## 2. hygiene-check on real patches

The checker is the v0 rule set from `eval-proto/hygiene-check/checker.py`, unchanged.
We compare its flags against the human `HYG` secondary label (debug prints, repro scripts, temp or unrelated files in the diff).

In [ ]:
H = []
for r in rows:
    rep = check(r["agent_patch"], r["instance_id"])
    g = gym.get(r["instance_id"])
    grep_ = check(g["patch"], r["instance_id"]) if g else None
    H.append(dict(instance_id=r["instance_id"], primary=r["primary_v3"],
                  human_hyg="HYG" in r["secondary_v3"],
                  score=rep.score, junk=len(rep.junk_files)>0, debug=len(rep.debug_leftovers)>0,
                  tests=len(rep.touches_test_files)>0, only_junk=rep.only_junk,
                  gold_score=(grep_.score if grep_ else None)))
H = pd.DataFrame(H)
H["auto_hyg"] = H.junk | H.debug | H.only_junk   # the checker's notion of HYG (test-file touching reported separately)
tp=((H.auto_hyg)&(H.human_hyg)).sum(); fp=((H.auto_hyg)&(~H.human_hyg)).sum()
fn=((~H.auto_hyg)&(H.human_hyg)).sum(); tn=((~H.auto_hyg)&(~H.human_hyg)).sum()
prec=tp/max(1,tp+fp); rec=tp/max(1,tp+fn); acc=(tp+tn)/len(H)
po=acc; pe=((tp+fp)*(tp+fn)+(fn+tn)*(fp+tn))/len(H)**2; kappa=(po-pe)/(1-pe) if pe<1 else float('nan')
hyg_summary = dict(n=len(H), tp=int(tp), fp=int(fp), fn=int(fn), tn=int(tn),
                   precision=round(prec,3), recall=round(rec,3), accuracy=round(acc,3), kappa=round(kappa,3),
                   agent_score_mean=round(H.score.mean(),1), agent_score_median=float(H.score.median()),
                   gold_score_mean=round(H.gold_score.dropna().mean(),1),
                   gold_score_100_share=round((H.gold_score.dropna()==100).mean(),3),
                   touches_tests_share=round(H.tests.mean(),3))
print(json.dumps(hyg_summary, indent=1))
print(H.groupby("primary").score.agg(["count","mean"]).round(1))
H.to_csv(f"{WORK}/hygiene_results.csv", index=False)
json.dump(hyg_summary, open(f"{WORK}/hygiene_summary.json","w"), indent=1)

## 3. localization-probe with Gemma 4

For every unique task instance we list all `.py` files in the repository at `base_commit` (blobless clone, no checkout),
retrieve the top-50 by BM25 over path tokens against the issue text, and ask Gemma to pick the 5 files most likely to need changes.
Ground truth = files touched by the gold patch. Metrics from `eval-proto/localization-probe/scorer.py`.

In [ ]:
REPOS = {}
def repo_dir(repo):
    d = f"/kaggle/temp/repos/{repo.replace('/','__')}"
    if not os.path.exists(d):
        os.makedirs(os.path.dirname(d), exist_ok=True)
        t=time.time()
        subprocess.run(["git","clone","--filter=blob:none","--no-checkout","-q",f"https://github.com/{repo}.git",d],check=True)
        print(f"cloned {repo} in {time.time()-t:.0f}s")
    return d

def py_files(repo, commit):
    d = repo_dir(repo)
    out = subprocess.run(["git","-C",d,"ls-tree","-r","--name-only",commit],capture_output=True,text=True)
    if out.returncode != 0:
        subprocess.run(["git","-C",d,"fetch","-q","--filter=blob:none","origin",commit])
        out = subprocess.run(["git","-C",d,"ls-tree","-r","--name-only",commit],capture_output=True,text=True)
    return [f for f in out.stdout.splitlines() if f.endswith(".py")]

GOLD_RE = re.compile(r"^diff --git a/(.*?) b/", re.M)
inst_ids = sorted({r["instance_id"] for r in rows})
INST = []
for iid in inst_ids:
    g = gym[iid]
    files = py_files(g["repo"], g["base_commit"])
    gold = [f for f in dict.fromkeys(GOLD_RE.findall(g["patch"])) if f.endswith(".py")]
    INST.append(dict(instance_id=iid, repo=g["repo"], issue=g["problem_statement"], files=files, gold=gold))
print(len(INST), "unique instances; median repo .py files:", int(pd.Series([len(i['files']) for i in INST]).median()))

In [ ]:
from rank_bm25 import BM25Okapi
TOK = re.compile(r"[A-Za-z][A-Za-z0-9]+")
def toks(s):
    out=[]
    for t in TOK.findall(s):
        t=t.lower(); out.append(t)
        out += [p for p in re.split(r"_", t) if len(p)>2 and p!=t]
    return out
POOL = 50
for it in INST:
    docs = [toks(f.replace("/"," ").replace("."," ")) for f in it["files"]]
    bm = BM25Okapi(docs)
    sc = bm.get_scores(toks(it["issue"]))
    order = sorted(range(len(it["files"])), key=lambda i: -sc[i])
    it["bm25"] = [it["files"][i] for i in order]
    it["pool"] = it["bm25"][:POOL]
pool_recall = sum(any(g in it["pool"] for g in it["gold"]) for it in INST)/len(INST)
print(f"gold file inside BM25 top-{POOL} pool: {pool_recall:.3f}")

In [ ]:
import torch, kagglehub, gc
from transformers import AutoProcessor, AutoModelForCausalLM, BitsAndBytesConfig

CANDIDATE_HANDLES = {
  "gemma-4-e4b-it": ["google/gemma-4/transformers/gemma-4-e4b-it", "google/gemma-4/transformers/gemma-4-E4B-it"],
  "gemma-4-12b-it": ["google/gemma-4/transformers/gemma-4-12b-it/1", "google/gemma-4/transformers/gemma-4-12b-it/2",
                     "google/gemma-4/transformers/gemma-4-12b-it", "google/gemma-4/transformers/gemma-4-12b-it-qat-q4_0-unquantized"],
}
def load(name):
    last=None
    for h in CANDIDATE_HANDLES[name]:
        try:
            path = kagglehub.model_download(h); print("downloaded", h); break
        except Exception as e:
            print("handle failed:", h, repr(e)[:200]); last=e; path=None
    if path is None: raise last
    proc = AutoProcessor.from_pretrained(path)
    q = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.float16)
    model = AutoModelForCausalLM.from_pretrained(path, quantization_config=q, device_map="auto", dtype=torch.float16)
    model.eval(); return proc, model, path

SYS = "You are an expert software engineer doing fault localization. You never edit code."
def prompt(it):
    cand = "\n".join(it["pool"])
    issue = it["issue"][:6000]
    return [{"role":"system","content":SYS},
            {"role":"user","content":
             f"Repository: {it['repo']}\n\nIssue:\n{issue}\n\nCandidate files (repository-relative):\n{cand}\n\n"
             "Which of the candidate files most likely need to be modified to resolve the issue? "
             "Answer with a JSON list of up to 5 paths copied exactly from the candidate list, most likely first. Output only the JSON list."}]

def parse(text, pool):
    m = re.search(r"\[.*?\]", text, re.S)
    picked=[]
    if m:
        try: picked=[p for p in json.loads(m.group(0)) if isinstance(p,str)]
        except Exception: picked=re.findall(r'"([^"]+\.py)"', m.group(0))
    if not picked: picked=re.findall(r"[\w/\.\-]+\.py", text)
    ranked=[p for p in dict.fromkeys(picked) if p in pool]
    return ranked + [p for p in pool if p not in ranked]   # fall back to BM25 order for the tail

In [ ]:
def run(name, batch=4, max_new=160):
    out_path = f"{WORK}/loc_{name}.jsonl"
    done = {}
    if os.path.exists(out_path):
        for l in open(out_path): d=json.loads(l); done[d["instance_id"]]=d
    todo=[it for it in INST if it["instance_id"] not in done]
    if not todo: return done
    proc, model, path = load(name); print("loaded", name, "from", path)
    proc.tokenizer.padding_side = "left"
    t0=time.time()
    with open(out_path,"a") as fo:
        for b in range(0,len(todo),batch):
            chunk=todo[b:b+batch]
            texts=[proc.apply_chat_template(prompt(it),tokenize=False,add_generation_prompt=True,enable_thinking=False) for it in chunk]
            enc=proc(text=texts,return_tensors="pt",padding=True).to(model.device)
            with torch.no_grad():
                gen=model.generate(**enc,max_new_tokens=max_new,do_sample=False)
            for it,seq in zip(chunk,gen):
                txt=proc.decode(seq[enc["input_ids"].shape[-1]:],skip_special_tokens=True)
                d=dict(instance_id=it["instance_id"], raw=txt, ranked=parse(txt,it["pool"])[:POOL])
                done[it["instance_id"]]=d; fo.write(json.dumps(d)+"\n"); fo.flush()
            if b % (batch*10) == 0: print(f"{name}: {len(done)}/{len(INST)}  {time.time()-t0:.0f}s")
    del model; gc.collect(); torch.cuda.empty_cache()
    return done

RESULTS = {}
for name in ["gemma-4-e4b-it", "gemma-4-12b-it"]:
    try:
        RESULTS[name] = run(name)
    except Exception as e:
        print("FAILED", name, repr(e)[:500])

## 4. Results

In [ ]:
lab_by_iid = collections.defaultdict(list)
for r in rows: lab_by_iid[r["instance_id"]].append(r["primary_v3"])
def evaluate(rank_fn, k):
    s=[score_instance(rank_fn(it), it["gold"], it["instance_id"], k=k) for it in INST]
    return s
systems = {"BM25 (paths)": lambda it: it["bm25"]}
for name,res in RESULTS.items():
    systems[name] = (lambda res: (lambda it: res[it["instance_id"]]["ranked"] if it["instance_id"] in res else it["pool"]))(res)
table=[]
for sname,fn in systems.items():
    row={"system":sname}
    for k in (1,3,5):
        agg=aggregate(evaluate(fn,k)); row[f"hit@{k}"]=round(agg["mean_hit@k"],3)
    row["MRR"]=round(aggregate(evaluate(fn,50))["mean_reciprocal_rank"],3)
    table.append(row)
loc_table=pd.DataFrame(table); print(loc_table.to_string(index=False))
print(f"\nn = {len(INST)} unique instances; gold-in-pool ceiling (top-{POOL}) = {pool_recall:.3f}")

# Does localization difficulty differ by the failure branch the GPT-4o agent fell into?
def branch(labs):
    if any(l.startswith("LOC") for l in labs): return "LOC-*"
    if any(l.startswith("NARROW") for l in labs): return "NARROW-*"
    return "other"
best = list(RESULTS)[-1] if RESULTS else None
by=[]
for it in INST:
    b=branch(lab_by_iid[it["instance_id"]])
    rec={"branch":b}
    for sname,fn in systems.items():
        rec[sname]=score_instance(fn(it),it["gold"],it["instance_id"],k=5).hit
    by.append(rec)
by=pd.DataFrame(by); by_branch=by.groupby("branch").agg(["count","mean"]).round(3)
print("\nhit@5 by the GPT-4o agent's failure branch:\n", by_branch)
loc_table.to_csv(f"{WORK}/localization_results.csv", index=False)
by.to_csv(f"{WORK}/localization_by_branch.csv", index=False)
json.dump(dict(loc=table, n=len(INST), pool=POOL, pool_recall=pool_recall,
               by_branch={b: {s: float(by[by.branch==b][s].mean()) for s in systems} | {"n": int((by.branch==b).sum())} for b in by.branch.unique()},
               hygiene=hyg_summary), open(f"{WORK}/summary.json","w"), indent=1)
print(open(f"{WORK}/summary.json").read())